# 03 — Synthetic vs Real Data Comparison

Side-by-side comparison of synthetic generated data against real deployment data.
Used to calibrate the generator so synthetic data is a realistic stand-in for real data during pipeline development.

**Sections:**
1. Load real data sample
2. Load synthetic data
3. Compare RSSI distributions
4. Compare multi-hub visibility rates
5. Compare temporal density and reporting cadence
6. Compare gateway clock offset behaviour
7. Calibration recommendations

> **Note:** Run `synthetic_data/generator.py` before running this notebook.

## Section 1 — Connection Setup

Credentials are loaded from a `.env` file in the repo root using `python-dotenv`.
This avoids hardcoding credentials and works reliably across terminal sessions
and VS Code notebook kernels.

**Setup (one time only):**
1. Create a file called `.env` in the `bio-tracker/` root (already gitignored)
2. Add these three lines with no spaces around `=` and no quotes:
```
DATABRICKS_SERVER_HOSTNAME=dbc-116b948a-d644.cloud.databricks.com
DATABRICKS_HTTP_PATH=/sql/1.0/warehouses/d89ac2793b1f6d5c
DATABRICKS_TOKEN=your_token_here
```
3. Install dotenv if needed: `pip install python-dotenv`

**Important:** Cell 1 must run before Cell 2. Always run cells top to bottom
or use "Run All" — never run the connection cell without running the dotenv
cell first.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file BEFORE any other imports
# .env lives in the repo root, one level up from analysis/
load_dotenv("../.env")

# Verify all three variables loaded correctly
print("HOSTNAME:", os.getenv("DATABRICKS_SERVER_HOSTNAME"))
print("HTTP_PATH:", os.getenv("DATABRICKS_HTTP_PATH"))
print("TOKEN set:", os.getenv("DATABRICKS_TOKEN") is not None)

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from databricks import sql

sys.path.append("..")
from data.synthetic_data.generator import generate
import config

OUTPUT_DIR = "../outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="darkgrid")

connection = sql.connect(
    server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN")
)

def run_query(query: str) -> pd.DataFrame:
    """Run a SQL query and return results as a pandas DataFrame."""
    with connection.cursor() as cursor:
        cursor.execute(query)
        return pd.DataFrame([row.asDict() for row in cursor.fetchall()])

print("Ready.")

## Section 1 — Load Real Data Sample

In [ ]:
# TODO: query one day of real data from Databricks
# Filter: gateway_id != test, rssi between -100 and 0
# Return standard internal format: gateway_id, timestamp, device_id, rssi
pass

## Section 2 — Load Synthetic Data

In [ ]:
# TODO: call generate() from data/synthetic_data/generator.py
# Print shape and head()
pass

## Section 3 — Compare RSSI Distributions

In [ ]:
# TODO: side-by-side histograms (or overlapping with alpha)
# Print: mean, std, median for both datasets
# Key question: does synthetic RSSI_REF and PATH_LOSS_EXP produce a 
# distribution that matches the real data mean (~-69 dBm) and spread?
pass

## Section 4 — Compare Multi-Hub Visibility Rates

In [ ]:
# TODO: for both datasets, compute % of (device, window) pairs with 2+ hubs
# Plot side-by-side bar charts of hub count distribution
# Key question: does the synthetic generator produce similar multi-hub
# visibility density to real data?
pass

## Section 5 — Compare Temporal Density and Reporting Cadence

In [ ]:
# TODO: plot readings per hour for both datasets over their respective time ranges
# Key question: is the synthetic cadence (SCAN_WINDOW_MINUTES=20) producing
# a similar density to real data?
pass

## Section 6 — Compare Gateway Clock Offset Behaviour

In [ ]:
# TODO: real data shows each gateway has a FIXED offset from the 10-min boundary
# Check whether the synthetic generator simulates this:
#   - Real: each hub always reports at e.g. :09, :19, :29 (fixed offset)
#   - Naive synthetic: all hubs report at same timestamp (wrong)
#   - Calibrated synthetic: each hub gets a random fixed offset assigned at creation
# Plot the same timeline strip chart as notebook 02 for synthetic data
# and compare visually
pass

## Section 7 — Calibration Recommendations

In [ ]:
# TODO: based on sections 3-6, print a summary of recommended changes to
# data/synthetic_data/config.py to better match real data:
#   - RSSI_REF adjustment (target mean ~-69 dBm)
#   - PATH_LOSS_EXP adjustment
#   - NOISE_STD adjustment
#   - Whether per-gateway clock offsets need to be added to generator.py
pass

In [ ]:
connection.close()
print("Connection closed.")